In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.transformer_baseline import (
    IndependentBandTransformerClassifier,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [4]:
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

split_path = (
    outputs_dir /
    "salinas_spatial_split_seed42.npz"
)

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

Train: 32337
Validation: 10952
Test: 10840


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GroupWiseSpectralEmbedding(nn.Module):
    """
    SpectralFormer-style group-wise spectral embedding.

    Converts groups of adjacent spectral bands into tokens.
    """

    def __init__(
        self,
        in_channels: int,
        embed_dim: int = 64,
        group_size: int = 3,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.embed_dim = embed_dim
        self.group_size = group_size

        self.proj = nn.Conv1d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=group_size,
            stride=group_size,
            padding=0,
        )

    def forward(self, x):
        # x: [B, C, H, W]

        # Spatial average -> spectral signature
        x = x.mean(dim=(-2, -1))

        # [B, C] -> [B, 1, C]
        x = x.unsqueeze(1)

        # [B, embed_dim, num_groups]
        x = self.proj(x)

        # [B, num_groups, embed_dim]
        x = x.transpose(1, 2)

        return x

In [6]:
class SpectralFormerEncoder(nn.Module):
    def __init__(
        self,
        embed_dim=64,
        depth=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth,
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.encoder(x)
        return self.norm(x)

In [7]:
class SpectralFormerClassifier(nn.Module):
    def __init__(
        self,
        in_channels,
        num_classes,
        embed_dim=64,
        depth=4,
        num_heads=4,
        group_size=3,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = GroupWiseSpectralEmbedding(
            in_channels=in_channels,
            embed_dim=embed_dim,
            group_size=group_size,
        )

        self.encoder = SpectralFormerEncoder(
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads,
            dropout=dropout,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):

        # [B, tokens, embed_dim]
        tokens = self.embedding(x)

        # Transformer
        encoded = self.encoder(tokens)

        # Global token pooling
        features = encoded.mean(dim=1)

        return self.head(features)

In [9]:
spectralformer_model = SpectralFormerClassifier(
    in_channels=204,
    num_classes=16,
    embed_dim=64,
    depth=4,
    num_heads=4,
    group_size=3,
    dropout=0.1,
).to(device)

print(spectralformer_model)

print(
    "Trainable parameters:",
    count_parameters(spectralformer_model)
)

SpectralFormerClassifier(
  (embedding): GroupWiseSpectralEmbedding(
    (proj): Conv1d(1, 64, kernel_size=(3,), stride=(3,))
  )
  (encoder): SpectralFormerEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (head): Sequential(
    (0): Lay

In [11]:
from dfm.data import *

In [18]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [19]:
train_dataset = SalinasPatchDataset(
    scene=scene,
    indices=train_indices,
    patch_size=15,
    normalize=True,
)

val_dataset = SalinasPatchDataset(
    scene=scene,
    indices=val_indices,
    patch_size=15,
    normalize=True,
)

test_dataset = SalinasPatchDataset(
    scene=scene,
    indices=test_indices,
    patch_size=15,
    normalize=True,
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

32337 10952 10840


In [20]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("DataLoaders ready.")

DataLoaders ready.


In [21]:
x, y = next(iter(train_loader))

print("Input :", x.shape)
print("Labels:", y.shape)
print("Sample label:", y[0].item())

Input : torch.Size([128, 204, 15, 15])
Labels: torch.Size([128])
Sample label: 2


In [22]:
x = x.to(device, dtype=torch.float32)

with torch.no_grad():
    logits = spectralformer_model(x)

print("Input :", x.shape)
print("Output:", logits.shape)

Input : torch.Size([128, 204, 15, 15])
Output: torch.Size([128, 16])


In [67]:
from pathlib import Path
import urllib.request

third_party_dir = PROJECT_ROOT / "third_party" / "SpectralFormer"
third_party_dir.mkdir(parents=True, exist_ok=True)

official_vit_path = third_party_dir / "vit_pytorch.py"

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/danfenghong/IEEE_TGRS_SpectralFormer/main/vit_pytorch.py",
    official_vit_path,
)

print("Downloaded:", official_vit_path)

Downloaded: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\third_party\SpectralFormer\vit_pytorch.py


In [68]:
%pip install einops -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    spectralformer_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)

In [24]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    total = 0

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total

In [25]:
def evaluate_model(model, loader, device):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for x, y in tqdm(loader, leave=False):
            x = x.to(device, dtype=torch.float32)

            logits = model(x)
            preds = logits.argmax(dim=1).cpu().numpy()

            y_pred.extend(preds)
            y_true.extend(y.numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    return accuracy, macro_f1, y_true, y_pred

In [26]:
from copy import deepcopy

EPOCHS = 50

best_val_f1 = -1.0
best_epoch = 0
best_state = None

history = []

for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        spectralformer_model,
        train_loader,
        criterion,
        optimizer,
        device,
    )

    val_acc, val_f1, _, _ = evaluate_model(
        spectralformer_model,
        val_loader,
        device,
    )

    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = deepcopy(
            spectralformer_model.state_dict()
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

print()
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_val_f1)

Epoch 01/50 | Loss: 0.7010 | Val Acc: 0.7950 | Val Macro-F1: 0.6448


Epoch 02/50 | Loss: 0.2808 | Val Acc: 0.8015 | Val Macro-F1: 0.7103


Epoch 03/50 | Loss: 0.2180 | Val Acc: 0.8693 | Val Macro-F1: 0.7442


Epoch 04/50 | Loss: 0.1862 | Val Acc: 0.9044 | Val Macro-F1: 0.7606


Epoch 05/50 | Loss: 0.1883 | Val Acc: 0.8944 | Val Macro-F1: 0.7855


...

Epoch 46/50 | Loss: 0.0241 | Val Acc: 0.9249 | Val Macro-F1: 0.8411


Epoch 47/50 | Loss: 0.0242 | Val Acc: 0.9225 | Val Macro-F1: 0.8401


Epoch 48/50 | Loss: 0.0218 | Val Acc: 0.9194 | Val Macro-F1: 0.8383


Epoch 49/50 | Loss: 0.0231 | Val Acc: 0.9197 | Val Macro-F1: 0.8389


Epoch 50/50 | Loss: 0.0224 | Val Acc: 0.9208 | Val Macro-F1: 0.8393

Best epoch: 44
Best validation Macro-F1: 0.8449427763445119


In [27]:
spectralformer_model.load_state_dict(best_state)

<All keys matched successfully>

In [28]:
test_acc, test_f1, y_test_spectralformer, y_pred_spectralformer = evaluate_model(
    spectralformer_model,
    test_loader,
    device,
)

print("=" * 55)
print("SPECTRALFORMER SPATIAL TEST RESULTS")
print("=" * 55)
print(f"Accuracy : {test_acc:.4f}")
print(f"Macro-F1 : {test_f1:.4f}")

SPECTRALFORMER SPATIAL TEST RESULTS
Accuracy : 0.9193
Macro-F1 : 0.8824


In [29]:
print(
    classification_report(
        y_test_spectralformer,
        y_pred_spectralformer,
        target_names=SALINAS_CLASS_NAMES,
        digits=4,
    )
)

                           precision    recall  f1-score   support

    Brocoli_green_weeds_1     1.0000    0.9296    0.9635       909
    Brocoli_green_weeds_2     0.8832    0.9719    0.9254       498
                   Fallow     0.9457    1.0000    0.9721       296
        Fallow_rough_plow     0.7564    1.0000    0.8613       295
            Fallow_smooth     1.0000    0.8656    0.9280       863
                  Stubble     1.0000    1.0000    1.0000       632
                   Celery     0.8054    1.0000    0.8922       120
         Grapes_untrained     0.9026    0.9307    0.9164      2899
     Soil_vinyard_develop     1.0000    1.0000    1.0000      1177
Corn_senesced_green_weeds     0.9804    0.5515    0.7059       544
      Lettuce_romaine_4wk     0.2339    1.0000    0.3792        51
      Lettuce_romaine_5wk     0.9912    1.0000    0.9956       448
      Lettuce_romaine_6wk     0.9357    1.0000    0.9668       233
      Lettuce_romaine_7wk     0.9378    0.8596    0.8970     

In [30]:
cm_spectralformer = confusion_matrix(
    y_test_spectralformer,
    y_pred_spectralformer,
)

cm_spectralformer_df = pd.DataFrame(
    cm_spectralformer,
    index=SALINAS_CLASS_NAMES,
    columns=SALINAS_CLASS_NAMES,
)

display(cm_spectralformer_df)

,Brocoli_green_weeds_1,Brocoli_green_weeds_2,Fallow,Fallow_rough_plow,Fallow_smooth,Stubble,Celery,Grapes_untrained,Soil_vinyard_develop,Corn_senesced_green_weeds,Lettuce_romaine_4wk,Lettuce_romaine_5wk,Lettuce_romaine_6wk,Lettuce_romaine_7wk,Vinyard_untrained,Vinyard_vertical_trellis
Brocoli_green_weeds_1,845,64,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Brocoli_green_weeds_2,0,484,0,0,0,0,12,0,0,0,0,0,0,2,0,0
Fallow,0,0,296,0,0,0,0,0,0,0,0,0,0,0,0,0
Fallow_rough_plow,0,0,0,295,0,0,0,0,0,0,0,0,0,0,0,0
Fallow_smooth,0,0,0,95,747,0,0,21,0,0,0,0,0,0,0,0
Stubble,0,0,0,0,0,632,0,0,0,0,0,0,0,0,0,0
Celery,0,0,0,0,0,0,120,0,0,0,0,0,0,0,0,0
Grapes_untrained,0,0,0,0,0,0,17,2698,0,0,0,0,0,11,173,0
Soil_vinyard_develop,0,0,0,0,0,0,0,0,1177,0,0,0,0,0,0,0
Corn_senesced_green_weeds,0,0,17,0,0,0,0,56,0,300,167,4,0,0,0,0


In [31]:
spectralformer_results = pd.DataFrame([
    {
        "model": "SpectralFormer",
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
    }
])

spectralformer_results_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_spatial_results.csv"
)

spectralformer_results.to_csv(
    spectralformer_results_path,
    index=False,
)

display(spectralformer_results)

,model,test_accuracy,test_macro_f1
0,SpectralFormer,0.91928,0.882376


In [32]:
np.save(
    outputs_dir / "spectralformer_spatial_y_test.npy",
    y_test_spectralformer,
)

np.save(
    outputs_dir / "spectralformer_spatial_y_pred.npy",
    y_pred_spectralformer,
)

In [33]:
cm_spectralformer_df.to_csv(
    outputs_dir / "spectralformer_spatial_confusion_matrix.csv"
)

In [34]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

param_count = count_parameters(spectralformer_model)

print(f"Trainable parameters: {param_count:,}")
print(f"Parameters (M): {param_count / 1e6:.4f}")

Trainable parameters: 201,488
Parameters (M): 0.2015


In [36]:
from pathlib import Path
import os

spectralformer_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_spatial_best.pt"
)

print("Exists:", spectralformer_path.exists())
print("Path:", spectralformer_path)

Exists: False
Path: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\spectralformer_spatial_best.pt


In [37]:
print(list((PROJECT_ROOT / "outputs" / "salinas").glob("*spectralformer*")))

[WindowsPath('c:/Users/Dines/Documents/Codex/2026-05-15/files-mentioned-by-the-user-dataset0/dense-forest-monitoring/outputs/salinas/spectralformer_spatial_confusion_matrix.csv'), WindowsPath('c:/Users/Dines/Documents/Codex/2026-05-15/files-mentioned-by-the-user-dataset0/dense-forest-monitoring/outputs/salinas/spectralformer_spatial_results.csv'), WindowsPath('c:/Users/Dines/Documents/Codex/2026-05-15/files-mentioned-by-the-user-dataset0/dense-forest-monitoring/outputs/salinas/spectralformer_spatial_y_pred.npy'), WindowsPath('c:/Users/Dines/Documents/Codex/2026-05-15/files-mentioned-by-the-user-dataset0/dense-forest-monitoring/outputs/salinas/spectralformer_spatial_y_test.npy')]


In [38]:
spectralformer_model.load_state_dict(best_state)
spectralformer_model.eval()

SpectralFormerClassifier(
  (embedding): GroupWiseSpectralEmbedding(
    (proj): Conv1d(1, 64, kernel_size=(3,), stride=(3,))
  )
  (encoder): SpectralFormerEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (head): Sequential(
    (0): Lay

In [39]:
spectralformer_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_spatial_best.pt"
)

torch.save(
    spectralformer_model.state_dict(),
    spectralformer_path,
)

print("Saved:", spectralformer_path)
print("Exists:", spectralformer_path.exists())

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\spectralformer_spatial_best.pt
Exists: True


In [40]:
import os

model_size_bytes = os.path.getsize(spectralformer_path)
model_size_mb = model_size_bytes / (1024 ** 2)

print(f"Model size: {model_size_mb:.3f} MB")

Model size: 0.790 MB


In [43]:
from thop import profile

spectralformer_model.eval()

dummy_input = torch.randn(
    1, 204, 15, 15,
    device=device,
)

flops, params = profile(
    spectralformer_model,
    inputs=(dummy_input,),
    verbose=False,
)

print(f"FLOPs: {flops:,}")
print(f"GFLOPs: {flops / 1e9:.4f}")

FLOPs: 9,083,904.0
GFLOPs: 0.0091


In [44]:
spectralformer_model.eval()

with torch.no_grad():
    for _ in range(20):
        _ = spectralformer_model(x)

torch.cuda.synchronize()

In [45]:
import time
import numpy as np

latencies = []

with torch.no_grad():
    for _ in range(100):
        torch.cuda.synchronize()
        start = time.perf_counter()

        _ = spectralformer_model(x)

        torch.cuda.synchronize()
        end = time.perf_counter()

        latencies.append((end - start) * 1000)

latencies = np.asarray(latencies)

mean_batch_latency = latencies.mean()
std_batch_latency = latencies.std()

print(f"Mean batch latency: {mean_batch_latency:.3f} ms")
print(f"Latency std: {std_batch_latency:.3f} ms")

Mean batch latency: 17.186 ms
Latency std: 20.077 ms


In [46]:
batch_size = x.shape[0]

mean_sample_latency = mean_batch_latency / batch_size

throughput = batch_size / (
    mean_batch_latency / 1000
)

print(f"Latency/sample: {mean_sample_latency:.4f} ms")
print(f"Throughput: {throughput:.2f} samples/sec")

Latency/sample: 0.1343 ms
Throughput: 7448.02 samples/sec


In [48]:
# RTX 2050 peak inference memory

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

spectralformer_model.eval()

x, _ = next(iter(test_loader))
x = x.to(device, dtype=torch.float32)

with torch.no_grad():
    _ = spectralformer_model(x)

torch.cuda.synchronize()

peak_memory = torch.cuda.max_memory_allocated(device)

print(
    f"Peak GPU memory: "
    f"{peak_memory / (1024 ** 2):.2f} MB"
)

Peak GPU memory: 60.73 MB


In [49]:
spectralformer_metrics = pd.DataFrame([{
    "model": "SpectralFormer",
    "accuracy": test_acc,
    "macro_f1": test_f1,
    "parameters": param_count,
    "parameters_m": param_count / 1e6,
    "model_size_mb": model_size_mb,
    "flops": flops,
    "gflops": flops / 1e9,
    "peak_gpu_memory_mb": peak_memory / (1024 ** 2),
    "latency_batch_ms": mean_batch_latency,
    "latency_batch_std_ms": std_batch_latency,
    "latency_per_sample_ms": mean_sample_latency,
    "throughput_samples_sec": throughput,
}])

display(spectralformer_metrics)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,latency_batch_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer,0.91928,0.882376,201488,0.201488,0.789824,9083904.0,0.009084,60.731934,17.185774,20.077257,0.134264,7448.020671


In [50]:
spectralformer_metrics.to_csv(
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_spatial_efficiency.csv",
    index=False,
)

In [52]:
from dfm.models.hybrid import HybridSpatialSpectralClassifier
from dfm.training.metrics import accuracy_score, macro_f1_score
from dfm.training.profiling import count_parameters

In [54]:
hybrid_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "hybrid_spatial_best.pt"
)

checkpoint = torch.load(
    hybrid_path,
    map_location=device,
    weights_only=False,
)

print(checkpoint.keys())

dict_keys(['model_state_dict', 'best_epoch', 'best_val_macro_f1', 'class_names', 'patch_size', 'bands', 'seed'])


In [55]:
hybrid_model = HybridSpatialSpectralClassifier(
    in_channels=204,
    num_classes=16,
    feature_dim=128,
    spectral_depth=2,
    spectral_heads=4,
    fusion="gated",
    dropout=0.1,
    max_bands=256,
).to(device)

hybrid_model.load_state_dict(
    checkpoint["model_state_dict"]
)

hybrid_model.eval()

print("Loaded Hybrid checkpoint successfully.")
print("Best epoch:", checkpoint["best_epoch"])
print("Best validation Macro-F1:", checkpoint["best_val_macro_f1"])

Loaded Hybrid checkpoint successfully.
Best epoch: 46
Best validation Macro-F1: 0.9563906490258482


In [56]:
hybrid_test_acc, hybrid_test_f1, y_test_hybrid, y_pred_hybrid = evaluate_model(
    hybrid_model,
    test_loader,
    device,
)

print("=" * 55)
print("HYBRID SPATIAL TEST RESULTS")
print("=" * 55)
print(f"Accuracy : {hybrid_test_acc:.4f}")
print(f"Macro-F1 : {hybrid_test_f1:.4f}")

HYBRID SPATIAL TEST RESULTS
Accuracy : 0.9810
Macro-F1 : 0.9639


In [57]:
import os

hybrid_size_bytes = os.path.getsize(hybrid_path)
hybrid_size_mb = hybrid_size_bytes / (1024 ** 2)

print(f"Model size: {hybrid_size_mb:.3f} MB")

Model size: 2.330 MB


In [58]:
hybrid_param_count = count_parameters(hybrid_model)

print(f"Trainable parameters: {hybrid_param_count:,}")
print(f"Parameters (M): {hybrid_param_count / 1e6:.4f}")

Trainable parameters: 605,712
Parameters (M): 0.6057


In [59]:
from thop import profile

hybrid_model.eval()

dummy_input = torch.randn(
    1,
    204,
    15,
    15,
    device=device,
)

hybrid_flops, hybrid_params = profile(
    hybrid_model,
    inputs=(dummy_input,),
    verbose=False,
)

print(f"FLOPs : {hybrid_flops:,}")
print(f"GFLOPs: {hybrid_flops / 1e9:.6f}")

FLOPs : 82,053,120.0
GFLOPs: 0.082053


In [60]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

hybrid_model.eval()

x, _ = next(iter(test_loader))
x = x.to(device, dtype=torch.float32)

with torch.no_grad():
    _ = hybrid_model(x)

torch.cuda.synchronize()

hybrid_peak_memory = torch.cuda.max_memory_allocated(device)

print(
    f"Peak GPU memory: "
    f"{hybrid_peak_memory / (1024 ** 2):.2f} MB"
)

Peak GPU memory: 156.16 MB


In [61]:
hybrid_model.eval()

with torch.no_grad():
    for _ in range(20):
        _ = hybrid_model(x)

torch.cuda.synchronize()

print("Warm-up complete.")

Warm-up complete.


In [62]:
import time
import numpy as np

hybrid_latencies = []

with torch.no_grad():
    for _ in range(100):

        torch.cuda.synchronize()
        start = time.perf_counter()

        _ = hybrid_model(x)

        torch.cuda.synchronize()
        end = time.perf_counter()

        hybrid_latencies.append(
            (end - start) * 1000
        )

hybrid_latencies = np.asarray(
    hybrid_latencies
)

hybrid_mean_batch_latency = hybrid_latencies.mean()
hybrid_std_batch_latency = hybrid_latencies.std()

print(
    f"Mean batch latency: "
    f"{hybrid_mean_batch_latency:.3f} ms"
)

print(
    f"Latency std: "
    f"{hybrid_std_batch_latency:.3f} ms"
)

Mean batch latency: 42.779 ms
Latency std: 16.950 ms


In [63]:
hybrid_batch_size = x.shape[0]

hybrid_latency_per_sample = (
    hybrid_mean_batch_latency /
    hybrid_batch_size
)

hybrid_throughput = (
    hybrid_batch_size /
    (hybrid_mean_batch_latency / 1000)
)

print(
    f"Latency/sample: "
    f"{hybrid_latency_per_sample:.4f} ms"
)

print(
    f"Throughput: "
    f"{hybrid_throughput:.2f} samples/sec"
)

Latency/sample: 0.3342 ms
Throughput: 2992.11 samples/sec


In [64]:
hybrid_metrics = pd.DataFrame([{
    "model": "Hybrid Spatial-Spectral",

    "accuracy": hybrid_test_acc,
    "macro_f1": hybrid_test_f1,

    "parameters": hybrid_param_count,
    "parameters_m": hybrid_param_count / 1e6,

    "model_size_mb": hybrid_size_mb,

    "flops": hybrid_flops,
    "gflops": hybrid_flops / 1e9,

    "peak_gpu_memory_mb":
        hybrid_peak_memory / (1024 ** 2),

    "latency_batch_ms":
        hybrid_mean_batch_latency,

    "latency_batch_std_ms":
        hybrid_std_batch_latency,

    "latency_per_sample_ms":
        hybrid_latency_per_sample,

    "throughput_samples_sec":
        hybrid_throughput,
}])

display(hybrid_metrics)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,latency_batch_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,Hybrid Spatial-Spectral,0.980996,0.963905,605712,0.605712,2.330137,82053120.0,0.082053,156.163574,42.779184,16.949881,0.334212,2992.109433


In [65]:
hybrid_efficiency_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "hybrid_spatial_efficiency.csv"
)

hybrid_metrics.to_csv(
    hybrid_efficiency_path,
    index=False,
)

print("Saved:", hybrid_efficiency_path)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\hybrid_spatial_efficiency.csv


In [66]:
comparison = pd.concat(
    [
        spectralformer_metrics,
        hybrid_metrics,
    ],
    ignore_index=True,
)

display(comparison)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,latency_batch_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer,0.919280,0.882376,201488,0.201488,0.789824,9083904.0,0.009084,60.731934,17.185774,20.077257,0.134264,7448.020671
1,Hybrid Spatial-Spectral,0.980996,0.963905,605712,0.605712,2.330137,82053120.0,0.082053,156.163574,42.779184,16.949881,0.334212,2992.109433


In [69]:
import sys

if str(third_party_dir) not in sys.path:
    sys.path.insert(0, str(third_party_dir))

from vit_pytorch import ViT

print(ViT)

<class 'vit_pytorch.ViT'>


In [70]:
def official_band_grouping(
    patch_chw,
    band_patch=3,
):
    """
    Reproduce SpectralFormer's official band-neighborhood
    grouping for a single [C, H, W] patch.

    Input:
        [bands, H, W]

    Output:
        [bands, H*W*band_patch]
    """

    C, H, W = patch_chw.shape

    if band_patch % 2 == 0:
        raise ValueError("band_patch must be odd.")

    x = patch_chw.transpose(1, 2, 0)
    # [H, W, C]

    x = x.reshape(H * W, C)
    # [H*W, C]

    half = band_patch // 2

    grouped = np.zeros(
        (H * W * band_patch, C),
        dtype=np.float32,
    )

    # Center spectral group
    grouped[
        half * H * W:
        (half + 1) * H * W,
        :
    ] = x

    # Left spectral neighbors
    for i in range(half):

        start = i * H * W
        end = (i + 1) * H * W

        grouped[start:end, :C - i - 1] = x[:, i + 1:]
        grouped[start:end, C - i - 1:] = x[:, :i + 1]

    # Right spectral neighbors
    for i in range(half):

        start = (half + i + 1) * H * W
        end = (half + i + 2) * H * W

        grouped[start:end, :C - i - 1] = x[:, i + 1:]
        grouped[start:end, C - i - 1:] = x[:, :i + 1]

    return grouped.transpose(1, 0).astype(np.float32)

In [71]:
sample_x, sample_y = train_dataset[0]

print("Original:", sample_x.shape)

grouped_x = official_band_grouping(
    sample_x,
    band_patch=3,
)

print("Official GSE input:", grouped_x.shape)

Original: (204, 15, 15)
Official GSE input: (204, 675)


In [72]:
class SpectralFormerSalinasDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        base_dataset,
        band_patch=3,
    ):
        self.base_dataset = base_dataset
        self.band_patch = band_patch

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):

        x, y = self.base_dataset[index]

        x = official_band_grouping(
            x,
            band_patch=self.band_patch,
        )

        return (
            torch.from_numpy(x),
            torch.tensor(y, dtype=torch.long),
        )

In [73]:
BAND_PATCHES = 3

spectralformer_train_dataset = (
    SpectralFormerSalinasDataset(
        train_dataset,
        band_patch=BAND_PATCHES,
    )
)

spectralformer_val_dataset = (
    SpectralFormerSalinasDataset(
        val_dataset,
        band_patch=BAND_PATCHES,
    )
)

spectralformer_test_dataset = (
    SpectralFormerSalinasDataset(
        test_dataset,
        band_patch=BAND_PATCHES,
    )
)

In [74]:
BATCH_SIZE = 64

spectralformer_train_loader = DataLoader(
    spectralformer_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

spectralformer_val_loader = DataLoader(
    spectralformer_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

spectralformer_test_loader = DataLoader(
    spectralformer_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [75]:
x_sf, y_sf = next(
    iter(spectralformer_train_loader)
)

print("Input :", x_sf.shape)
print("Labels:", y_sf.shape)

Input : torch.Size([64, 204, 675])
Labels: torch.Size([64])


In [76]:
spectralformer_model = ViT(
    image_size=15,
    near_band=3,
    num_patches=204,
    num_classes=16,

    dim=64,
    depth=5,
    heads=4,
    mlp_dim=8,

    dropout=0.1,
    emb_dropout=0.1,

    mode="CAF",
).to(device)

In [77]:
print(spectralformer_model)

print(
    "Trainable parameters:",
    count_parameters(spectralformer_model),
)

print(
    "Parameters (M):",
    count_parameters(spectralformer_model) / 1e6,
)

ViT(
  (patch_to_embedding): Linear(in_features=675, out_features=64, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-4): 5 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=64, out_features=192, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=64, out_features=64, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=64, out_features=8, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=0.1, inplace=False)


In [78]:
x_sf = x_sf.to(
    device=device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits_sf = spectralformer_model(x_sf)

print("Input :", x_sf.shape)
print("Output:", logits_sf.shape)

Input : torch.Size([64, 204, 675])
Output: torch.Size([64, 16])


In [79]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    spectralformer_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)

In [80]:
def train_spectralformer_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    running_loss = 0.0
    total = 0

    for x, y in tqdm(loader, leave=False):

        x = x.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)

        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total

In [81]:
def evaluate_spectralformer(
    model,
    loader,
    device,
):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for x, y in tqdm(loader, leave=False):

            x = x.to(
                device,
                dtype=torch.float32,
            )

            logits = model(x)

            preds = (
                logits
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            y_pred.extend(preds)
            y_true.extend(y.numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    return (
        accuracy,
        macro_f1,
        y_true,
        y_pred,
    )

In [82]:
from copy import deepcopy

EPOCHS = 50

best_val_f1 = -1.0
best_epoch = 0
best_state = None

spectralformer_history = []

for epoch in range(1, EPOCHS + 1):

    train_loss = train_spectralformer_epoch(
        spectralformer_model,
        spectralformer_train_loader,
        criterion,
        optimizer,
        device,
    )

    val_acc, val_f1, _, _ = evaluate_spectralformer(
        spectralformer_model,
        spectralformer_val_loader,
        device,
    )

    scheduler.step()

    spectralformer_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch

        best_state = deepcopy(
            spectralformer_model.state_dict()
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

print()
print("Best epoch:", best_epoch)
print(
    "Best validation Macro-F1:",
    best_val_f1,
)

Epoch 01/50 | Loss: 0.3283 | Val Acc: 0.8851 | Val Macro-F1: 0.8470


Epoch 02/50 | Loss: 0.0604 | Val Acc: 0.8966 | Val Macro-F1: 0.7895


Epoch 03/50 | Loss: 0.0316 | Val Acc: 0.9120 | Val Macro-F1: 0.8301


Epoch 04/50 | Loss: 0.0209 | Val Acc: 0.9172 | Val Macro-F1: 0.8674


Epoch 05/50 | Loss: 0.0176 | Val Acc: 0.9368 | Val Macro-F1: 0.8649


...

Epoch 46/50 | Loss: 0.0000 | Val Acc: 0.9289 | Val Macro-F1: 0.8609


Epoch 47/50 | Loss: 0.0000 | Val Acc: 0.9290 | Val Macro-F1: 0.8613


Epoch 48/50 | Loss: 0.0000 | Val Acc: 0.9285 | Val Macro-F1: 0.8609


Epoch 49/50 | Loss: 0.0000 | Val Acc: 0.9289 | Val Macro-F1: 0.8620


Epoch 50/50 | Loss: 0.0000 | Val Acc: 0.9288 | Val Macro-F1: 0.8619

Best epoch: 9
Best validation Macro-F1: 0.8940662935881525


In [83]:
spectralformer_model.load_state_dict(
    best_state
)

spectralformer_model.eval()

ViT(
  (patch_to_embedding): Linear(in_features=675, out_features=64, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-4): 5 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=64, out_features=192, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=64, out_features=64, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=64, out_features=8, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=0.1, inplace=False)


In [84]:
spectralformer_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_official_spatial_best.pt"
)

torch.save(
    {
        "model_state_dict":
            spectralformer_model.state_dict(),

        "best_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_val_f1,

        "class_names":
            SALINAS_CLASS_NAMES,

        "patch_size":
            15,

        "bands":
            204,

        "band_patches":
            3,

        "mode":
            "CAF",

        "seed":
            42,
    },
    spectralformer_path,
)

print("Saved:", spectralformer_path)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\spectralformer_official_spatial_best.pt


In [85]:
spectralformer_history_df = pd.DataFrame(
    spectralformer_history
)

history_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_official_spatial_history.csv"
)

spectralformer_history_df.to_csv(
    history_path,
    index=False,
)

display(
    spectralformer_history_df.tail()
)

,epoch,train_loss,val_accuracy,val_macro_f1
45,46,0.000007,0.928871,0.860896
46,47,0.000007,0.928963,0.861332
47,48,0.000007,0.928506,0.860933
48,49,0.000012,0.928871,0.861970
49,50,0.000012,0.928780,0.861931


In [86]:
(
    sf_test_acc,
    sf_test_f1,
    y_test_sf,
    y_pred_sf,
) = evaluate_spectralformer(
    spectralformer_model,
    spectralformer_test_loader,
    device,
)

print("=" * 60)
print("OFFICIAL SPECTRALFORMER SPATIAL TEST RESULTS")
print("=" * 60)
print(f"Accuracy : {sf_test_acc:.4f}")
print(f"Macro-F1 : {sf_test_f1:.4f}")

OFFICIAL SPECTRALFORMER SPATIAL TEST RESULTS
Accuracy : 0.9274
Macro-F1 : 0.9187


In [87]:
x_sf, _ = next(iter(spectralformer_test_loader))
x_sf = x_sf.to(device, dtype=torch.float32)

print(x_sf.shape)

torch.Size([64, 204, 675])


In [88]:
import os

spectralformer_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_official_spatial_best.pt"
)

print("Exists:", spectralformer_path.exists())

sf_size_bytes = os.path.getsize(spectralformer_path)
sf_size_mb = sf_size_bytes / (1024 ** 2)

print(f"Checkpoint size: {sf_size_mb:.3f} MB")

Exists: True
Checkpoint size: 1.553 MB


In [89]:
sf_param_count = count_parameters(spectralformer_model)

print(f"Trainable parameters: {sf_param_count:,}")
print(f"Parameters (M): {sf_param_count / 1e6:.4f}")

Trainable parameters: 399,381
Parameters (M): 0.3994


In [90]:
from thop import profile

spectralformer_model.eval()

dummy_sf = torch.randn(
    1,
    204,
    675,
    device=device,
)

sf_flops, sf_profile_params = profile(
    spectralformer_model,
    inputs=(dummy_sf,),
    verbose=False,
)

print(f"FLOPs : {sf_flops:,}")
print(f"GFLOPs: {sf_flops / 1e9:.6f}")

FLOPs : 43,319,680.0
GFLOPs: 0.043320


In [91]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

spectralformer_model.eval()

x_sf, _ = next(iter(spectralformer_test_loader))

x_sf = x_sf.to(
    device,
    dtype=torch.float32,
)

with torch.no_grad():
    _ = spectralformer_model(x_sf)

torch.cuda.synchronize()

sf_peak_memory = torch.cuda.max_memory_allocated(device)

print(
    f"Peak GPU memory: "
    f"{sf_peak_memory / (1024 ** 2):.2f} MB"
)

Peak GPU memory: 215.10 MB


In [92]:
with torch.no_grad():
    for _ in range(20):
        _ = spectralformer_model(x_sf)

torch.cuda.synchronize()

print("Warm-up complete.")

Warm-up complete.


In [93]:
import time
import numpy as np

sf_latencies = []

with torch.no_grad():
    for _ in range(100):

        torch.cuda.synchronize()
        start = time.perf_counter()

        _ = spectralformer_model(x_sf)

        torch.cuda.synchronize()
        end = time.perf_counter()

        sf_latencies.append(
            (end - start) * 1000
        )

sf_latencies = np.asarray(sf_latencies)

sf_mean_batch_latency = sf_latencies.mean()
sf_std_batch_latency = sf_latencies.std()

print(
    f"Mean batch latency: "
    f"{sf_mean_batch_latency:.3f} ms"
)

print(
    f"Latency std: "
    f"{sf_std_batch_latency:.3f} ms"
)

Mean batch latency: 39.389 ms
Latency std: 27.540 ms


In [94]:
sf_batch_size = x_sf.shape[0]

sf_latency_per_sample = (
    sf_mean_batch_latency /
    sf_batch_size
)

sf_throughput = (
    sf_batch_size /
    (sf_mean_batch_latency / 1000)
)

print(
    f"Latency/sample: "
    f"{sf_latency_per_sample:.4f} ms"
)

print(
    f"Throughput: "
    f"{sf_throughput:.2f} samples/sec"
)

Latency/sample: 0.6155 ms
Throughput: 1624.80 samples/sec


In [95]:
official_sf_metrics = pd.DataFrame([{
    "model": "SpectralFormer (Official)",

    "accuracy": sf_test_acc,
    "macro_f1": sf_test_f1,

    "parameters": sf_param_count,
    "parameters_m": sf_param_count / 1e6,

    "model_size_mb": sf_size_mb,

    "flops": sf_flops,
    "gflops": sf_flops / 1e9,

    "peak_gpu_memory_mb":
        sf_peak_memory / (1024 ** 2),

    "latency_batch_ms":
        sf_mean_batch_latency,

    "latency_batch_std_ms":
        sf_std_batch_latency,

    "latency_per_sample_ms":
        sf_latency_per_sample,

    "throughput_samples_sec":
        sf_throughput,
}])

display(official_sf_metrics)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,latency_batch_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer (Official),0.927399,0.918674,399381,0.399381,1.552929,43319680.0,0.04332,215.10498,39.389343,27.540425,0.615458,1624.804963


In [96]:
official_sf_metrics.to_csv(
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_official_spatial_efficiency.csv",
    index=False,
)

print("Official SpectralFormer efficiency results saved.")

Official SpectralFormer efficiency results saved.


In [97]:
final_sf_hybrid_comparison = pd.concat(
    [
        official_sf_metrics,
        hybrid_metrics,
    ],
    ignore_index=True,
)

display(final_sf_hybrid_comparison)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,latency_batch_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer (Official),0.927399,0.918674,399381,0.399381,1.552929,43319680.0,0.043320,215.104980,39.389343,27.540425,0.615458,1624.804963
1,Hybrid Spatial-Spectral,0.980996,0.963905,605712,0.605712,2.330137,82053120.0,0.082053,156.163574,42.779184,16.949881,0.334212,2992.109433


In [98]:
SPECTRALFORMER_BENCH_BATCH = 128

spectralformer_bench_loader = DataLoader(
    spectralformer_test_dataset,
    batch_size=SPECTRALFORMER_BENCH_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

x_sf, _ = next(iter(spectralformer_bench_loader))

x_sf = x_sf.to(
    device,
    dtype=torch.float32,
)

print("Benchmark input:", x_sf.shape)

Benchmark input: torch.Size([128, 204, 675])


In [99]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

spectralformer_model.eval()

with torch.no_grad():
    _ = spectralformer_model(x_sf)

torch.cuda.synchronize()

sf_peak_memory_128 = torch.cuda.max_memory_allocated(device)

print(
    f"Peak GPU memory: "
    f"{sf_peak_memory_128 / (1024 ** 2):.2f} MB"
)

Peak GPU memory: 370.26 MB


In [100]:
with torch.no_grad():
    for _ in range(20):
        _ = spectralformer_model(x_sf)

torch.cuda.synchronize()

print("Warm-up complete.")

Warm-up complete.


In [101]:
import time
import numpy as np

sf_latencies_128 = []

with torch.no_grad():
    for _ in range(100):

        torch.cuda.synchronize()
        start = time.perf_counter()

        _ = spectralformer_model(x_sf)

        torch.cuda.synchronize()
        end = time.perf_counter()

        sf_latencies_128.append(
            (end - start) * 1000
        )

sf_latencies_128 = np.asarray(
    sf_latencies_128
)

sf_mean_batch_latency_128 = (
    sf_latencies_128.mean()
)

sf_std_batch_latency_128 = (
    sf_latencies_128.std()
)

sf_median_batch_latency_128 = (
    np.median(sf_latencies_128)
)

sf_p95_batch_latency_128 = (
    np.percentile(sf_latencies_128, 95)
)

print(
    f"Mean batch latency   : "
    f"{sf_mean_batch_latency_128:.3f} ms"
)

print(
    f"Median batch latency : "
    f"{sf_median_batch_latency_128:.3f} ms"
)

print(
    f"P95 batch latency    : "
    f"{sf_p95_batch_latency_128:.3f} ms"
)

print(
    f"Latency std          : "
    f"{sf_std_batch_latency_128:.3f} ms"
)

Mean batch latency   : 67.641 ms
Median batch latency : 65.251 ms
P95 batch latency    : 71.436 ms
Latency std          : 18.100 ms


In [102]:
sf_latency_per_sample_128 = (
    sf_mean_batch_latency_128 / 128
)

sf_throughput_128 = (
    128 /
    (sf_mean_batch_latency_128 / 1000)
)

print(
    f"Mean latency/sample: "
    f"{sf_latency_per_sample_128:.4f} ms"
)

print(
    f"Throughput: "
    f"{sf_throughput_128:.2f} samples/sec"
)

Mean latency/sample: 0.5284 ms
Throughput: 1892.35 samples/sec


In [103]:
official_sf_metrics_128 = pd.DataFrame([{
    "model": "SpectralFormer (Official)",

    "accuracy": sf_test_acc,
    "macro_f1": sf_test_f1,

    "parameters": sf_param_count,
    "parameters_m": sf_param_count / 1e6,

    "model_size_mb": sf_size_mb,

    "flops": sf_flops,
    "gflops": sf_flops / 1e9,

    "peak_gpu_memory_mb":
        sf_peak_memory_128 / (1024 ** 2),

    "batch_size": 128,

    "latency_batch_ms":
        sf_mean_batch_latency_128,

    "latency_batch_median_ms":
        sf_median_batch_latency_128,

    "latency_batch_p95_ms":
        sf_p95_batch_latency_128,

    "latency_batch_std_ms":
        sf_std_batch_latency_128,

    "latency_per_sample_ms":
        sf_latency_per_sample_128,

    "throughput_samples_sec":
        sf_throughput_128,
}])

display(official_sf_metrics_128)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer (Official),0.927399,0.918674,399381,0.399381,1.552929,43319680.0,0.04332,370.256348,128,67.640807,65.25065,71.43573,18.100158,0.528444,1892.348801


In [104]:
official_sf_metrics_128.to_csv(
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_official_spatial_efficiency.csv",
    index=False,
)

print("Saved corrected batch-128 benchmark.")

Saved corrected batch-128 benchmark.


In [105]:
final_comparison_128 = pd.concat(
    [
        official_sf_metrics_128,
        hybrid_metrics,
    ],
    ignore_index=True,
)

display(final_comparison_128)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer (Official),0.927399,0.918674,399381,0.399381,1.552929,43319680.0,0.043320,370.256348,128.0,67.640807,65.25065,71.43573,18.100158,0.528444,1892.348801
1,Hybrid Spatial-Spectral,0.980996,0.963905,605712,0.605712,2.330137,82053120.0,0.082053,156.163574,NaN,42.779184,NaN,NaN,16.949881,0.334212,2992.109433


In [106]:
HYBRID_BENCH_BATCH = 128

hybrid_bench_loader = DataLoader(
    test_dataset,
    batch_size=HYBRID_BENCH_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

x_hybrid, _ = next(iter(hybrid_bench_loader))
x_hybrid = x_hybrid.to(device, dtype=torch.float32)

print(x_hybrid.shape)

torch.Size([128, 204, 15, 15])


In [107]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

hybrid_model.eval()

with torch.no_grad():
    for _ in range(20):
        _ = hybrid_model(x_hybrid)

torch.cuda.synchronize()

hybrid_peak_memory_128 = torch.cuda.max_memory_allocated(device)

hybrid_latencies_128 = []

with torch.no_grad():
    for _ in range(100):
        torch.cuda.synchronize()
        start = time.perf_counter()

        _ = hybrid_model(x_hybrid)

        torch.cuda.synchronize()
        end = time.perf_counter()

        hybrid_latencies_128.append(
            (end - start) * 1000
        )

hybrid_latencies_128 = np.asarray(
    hybrid_latencies_128
)

hybrid_mean_latency_128 = hybrid_latencies_128.mean()
hybrid_median_latency_128 = np.median(hybrid_latencies_128)
hybrid_p95_latency_128 = np.percentile(
    hybrid_latencies_128,
    95,
)
hybrid_std_latency_128 = hybrid_latencies_128.std()

hybrid_latency_per_sample_128 = (
    hybrid_mean_latency_128 / 128
)

hybrid_throughput_128 = (
    128 / (hybrid_mean_latency_128 / 1000)
)

print(f"Peak memory : {hybrid_peak_memory_128 / (1024**2):.2f} MB")
print(f"Mean        : {hybrid_mean_latency_128:.3f} ms")
print(f"Median      : {hybrid_median_latency_128:.3f} ms")
print(f"P95         : {hybrid_p95_latency_128:.3f} ms")
print(f"Std         : {hybrid_std_latency_128:.3f} ms")
print(f"Per sample  : {hybrid_latency_per_sample_128:.4f} ms")
print(f"Throughput  : {hybrid_throughput_128:.2f} samples/s")

Peak memory : 252.45 MB
Mean        : 43.815 ms
Median      : 44.874 ms
P95         : 45.744 ms
Std         : 2.002 ms
Per sample  : 0.3423 ms
Throughput  : 2921.36 samples/s


In [108]:
hybrid_metrics_128 = pd.DataFrame([{
    "model": "Hybrid Spatial-Spectral",

    "accuracy": hybrid_test_acc,
    "macro_f1": hybrid_test_f1,

    "parameters": hybrid_param_count,
    "parameters_m": hybrid_param_count / 1e6,

    "model_size_mb": hybrid_size_mb,

    "flops": hybrid_flops,
    "gflops": hybrid_flops / 1e9,

    "peak_gpu_memory_mb":
        hybrid_peak_memory_128 / (1024 ** 2),

    "batch_size": 128,

    "latency_batch_ms":
        hybrid_mean_latency_128,

    "latency_batch_median_ms":
        hybrid_median_latency_128,

    "latency_batch_p95_ms":
        hybrid_p95_latency_128,

    "latency_batch_std_ms":
        hybrid_std_latency_128,

    "latency_per_sample_ms":
        hybrid_latency_per_sample_128,

    "throughput_samples_sec":
        hybrid_throughput_128,
}])

hybrid_efficiency_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "hybrid_spatial_efficiency.csv"
)

hybrid_metrics_128.to_csv(
    hybrid_efficiency_path,
    index=False,
)

print("Saved:", hybrid_efficiency_path)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\hybrid_spatial_efficiency.csv


In [109]:
print(
    pd.read_csv(hybrid_efficiency_path)
)

                     model  accuracy  macro_f1  parameters  parameters_m  \
0  Hybrid Spatial-Spectral  0.980996  0.963905      605712      0.605712   

   model_size_mb       flops    gflops  peak_gpu_memory_mb  batch_size  \
0       2.330137  82053120.0  0.082053          252.453613         128   

   latency_batch_ms  latency_batch_median_ms  latency_batch_p95_ms  \
0          43.81523                  44.8742              45.74441   

   latency_batch_std_ms  latency_per_sample_ms  throughput_samples_sec  
0              2.001661               0.342306             2921.358623  
